In [ ]:
# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then work from this notebook's own folder, which is
# what the relative paths below assume.
import os
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "requirements.txt").is_file() and _root != _root.parent:
    _root = _root.parent
_here = _root / "refinement-per-claim"
if not _here.is_dir():
    raise RuntimeError(
        "Could not locate " + str(_here) + ". Run this notebook from inside the "
        "cloned repository."
    )
os.chdir(_here)

# Imports

In [ ]:
import os
from smolagents import ToolCallingAgent, LiteLLMModel, PromptTemplates, tool, ActionStep, TaskStep, MessageRole
import pickle
import json
import pandas as pd
import numpy as np
import logging
from tqdm import tqdm
import ast
import time
import matplotlib.pyplot as plt
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()
# Set up logging

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
print("Anthropic API Key:", os.environ["ANTHROPIC_API_KEY"])

from pro_ukrainian_agents import *
from manager_agent import *
from mediator_agent import *
from pro_russian_agents_with_memory import *
from memory_summarizer_agent import *

# General Data for usage across all refinements

## Pro-Russian Claims and Users
### (Data is intentionally unavailable as discussed in paper)

In [ ]:
claims = pd.read_excel(r"../Data/Pro Russian top users and narratives.xlsx", sheet_name="top_20_narratives_20250313_1716")['description'].tolist()
df_pro_russian_users = pd.read_excel(r"../Data/pro_russian_users_data_for_agents.xlsx")

## Target KPIs for refinement

In [ ]:
dict_target_kpis = {
    "CN_Creator_Agent_1": "Persuasiveness",
    "CN_Creator_Agent_2": "Emotional Engagement",
    "CN_Creator_Agent_3": "Shareability"
}

## Helpful Function for Implementing Early Stopping

In [ ]:
def early_stopping_check(patience, best_kpi_score, best_kpi_score_iteration, current_kpi_score, current_iteration):
    flag = False
    if current_kpi_score > best_kpi_score:
        best_kpi_score = current_kpi_score
        best_kpi_score_iteration = current_iteration
    else:
        if current_iteration - best_kpi_score_iteration >= patience:
            flag = True

    return best_kpi_score, best_kpi_score_iteration, flag

# Example: Claim 1
### (Change `claim_str` and index inside `claims` to run a different claim)

In [ ]:
claim_str = "claim1"
claim = claims[0]
claim

### Pro-Ukrainian Agents

In [ ]:
file_path = f"Refinement_Per_Claim_Final/{claim_str}/system_prompts_old_combs.json"
if os.path.exists(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        pro_ukrainian_agents_system_prompts = json.load(f)
else:
    pro_ukrainian_agents_system_prompts = {
    "CN_Creator_Agent_1": system_prompt_agent_1,
    "CN_Creator_Agent_2": system_prompt_agent_2,
    "CN_Creator_Agent_3": system_prompt_agent_3
}
    
pro_ukrainian_agents_system_prompts

In [ ]:
pro_ukrainian_agents_type3 = create_pro_ukrainian_agents_claude(pro_ukrainian_agents_descriptions, pro_ukrainian_agents_system_prompts, temp=1.0)
pro_ukrainian_agents_type3

In [ ]:
lst_pro_ukrainian_agents_type3 = list(pro_ukrainian_agents_type3.values())
lst_pro_ukrainian_agents_type3

### Manager Agent

In [ ]:
manager_folder_path = f"Refinement_Per_Claim_Final/{claim_str}/Manager_Agent"
if os.path.exists(manager_folder_path):
    manager_agent_type3 = load_manager_agent(lst_pro_ukrainian_agents_type3, manager_description, manager_system_prompt, manager_folder_path)
else:
    manager_agent_type3 = create_manager_agent_claude(lst_pro_ukrainian_agents_type3, manager_description, manager_system_prompt)

### Saving Manager and pro-Ukrainian Agents

In [ ]:
managed_agents_folder_path = f"Refinement_Per_Claim_Final/{claim_str}/Manager_Agent/managed_agents"

In [ ]:
save_manager_agent(manager_agent_type3, manager_folder_path)
save_pro_ukrainian_agents(pro_ukrainian_agents_type3, managed_agents_folder_path)

### Mediator Agent

In [ ]:
mediator_agent = create_mediator_agent(mediator_system_prompt, mediator_description)
save_mediator_agent(mediator_agent)

### Memory Summarizer Agent

In [ ]:
memory_summarizer_agent = create_memory_summarizer_agent(memory_summarizer_agent_system_prompt, memory_summarizer_agent_description)
save_memory_summarizer_agent(memory_summarizer_agent)

### Refinement

In [ ]:
root_df_agent1_type3 = f"refinement_per_claim_final_results/{claim_str}/CN_Creator_Agent_1.csv"
root_df_agent2_type3 = f"refinement_per_claim_final_results/{claim_str}/CN_Creator_Agent_2.csv"
root_df_agent3_type3 = f"refinement_per_claim_final_results/{claim_str}/CN_Creator_Agent_3.csv"
columns = ["claim", "counter_narrative", "avg_pers", "avg_emot", "avg_share", "std_pers", "std_emot", "std_share", "aggregated_feedback", "updated_system_prompt"]

if os.path.exists(root_df_agent1_type3):
    df_claude_agent1_type3 = pd.read_csv(root_df_agent1_type3)
else:
    df_claude_agent1_type3 = pd.DataFrame(columns=columns)

if os.path.exists(root_df_agent2_type3):
    df_claude_agent2_type3 = pd.read_csv(root_df_agent2_type3)
else:
    df_claude_agent2_type3 = pd.DataFrame(columns=columns)

if os.path.exists(root_df_agent3_type3):
    df_claude_agent3_type3 = pd.read_csv(root_df_agent3_type3)
else:
    df_claude_agent3_type3 = pd.DataFrame(columns=columns)

In [ ]:
df_claude_agent1_type3

In [ ]:
dict_of_dfs = {
    "CN_Creator_Agent_1": df_claude_agent1_type3,
    "CN_Creator_Agent_2": df_claude_agent2_type3,
    "CN_Creator_Agent_3": df_claude_agent3_type3
}

In [ ]:
dict_of_flags = {
    "CN_Creator_Agent_1": False,
    "CN_Creator_Agent_2": False,
    "CN_Creator_Agent_3": False
}

In [ ]:
dict_last_iteration = {
    "CN_Creator_Agent_1": 0,
    "CN_Creator_Agent_2": 0,
    "CN_Creator_Agent_3": 0
} # This dictionary helps in case there is need to continue the refinement from a specific point

num_iterations = 25 # This variables holds how many iterations to perform starting from last iteration
patience = 8 # This variable holds how many iterations to wait for improvement before stopping the refinement

In [ ]:
# Defining helpful variables for early stopping
best_pers_score = 0
best_emot_score = 0
best_share_score = 0
best_pers_score_iteration = 0
best_emot_score_iteration = 0
best_share_score_iteration = 0

for agent_name, pro_ukrainian_agent in pro_ukrainian_agents_type3.items():
    # Create or load the pro-Russian Evaluators
    evaluator_agents_folder_path = f"Refinement_Per_Claim_Final/{claim_str}/Evaluator_Agents_For_{agent_name}"
    if os.path.exists(evaluator_agents_folder_path):
        pro_russian_agents_type3 = load_pro_russian_agents(df_pro_russian_users, 3, evaluator_agents_folder_path)
    else:
        pro_russian_agents_type3 = create_pro_russian_agents_claude(df_pro_russian_users, 3)
        save_pro_russian_agents(pro_russian_agents_type3, evaluator_agents_folder_path)

    for i in tqdm(range(dict_last_iteration[agent_name], num_iterations), desc=f"Iterations For {agent_name}"):
        pro_ukrainian_agent = pro_ukrainian_agents_type3[agent_name]
        # Check if early stopping flag is raised
        if dict_of_flags[agent_name]:
            break

        time.sleep(1)
        counter_narrative = pro_ukrainian_agent.run(claim)

        input_for_evaluators = f"""
pro-Russian claim: {claim}

pro-Ukrainian Counter-Narrative: {counter_narrative}
"""
        
        all_good_points = {"Persuasiveness": [], "Emotional_Engagement": [], "Shareability": []}
        all_bad_points  = {"Persuasiveness": [], "Emotional_Engagement": [], "Shareability": []}
        pers_scores = []
        emot_scores = []
        share_scores = []

        # Iterating over all evaluator agents for feedback
        for evaluator_name, evaluator_agent in pro_russian_agents_type3.items():
            flag = True
            j = 0
            while flag and (j < 3):
                try:
                    time.sleep(5)
                    response_str = evaluator_agent.run(input_for_evaluators, reset=False)
                    response_json = json.loads(response_str)
                    flag = False
                except:
                    print("Response was probably not in JSON format, trying again.")
                    j += 1
    
            # Collect good points
            for kpi in ["Persuasiveness", "Emotional_Engagement", "Shareability"]:
                all_good_points[kpi].extend(response_json["GoodPoints"][kpi])
            
            # Collect bad points
            for kpi in ["Persuasiveness", "Emotional_Engagement", "Shareability"]:
                all_bad_points[kpi].extend(response_json["BadPoints"][kpi])
                
            # Collect numeric scores
            pers_scores.append(response_json["Scores"]["Persuasiveness"])
            emot_scores.append(response_json["Scores"]["Emotional_Engagement"])
            share_scores.append(response_json["Scores"]["Shareability"])

            ### MEMORY SUMMARIZATION ###
            # First 5 Iterations
            if i == 4:
                input_for_summarizer = transform_memory(evaluator_agent.memory.steps, i - 3)

                time.sleep(5)
                generated_summary = memory_summarizer_agent.run(input_for_summarizer)

                evaluator_agent.memory.reset()
                evaluator_agent.memory.steps = [TaskStep(task=generated_summary, task_images=[])]
            
            # After 5 non-first Iterations
            if i != 4 and (i + 1) % 5 == 0:
                previous_summary = evaluator_agent.memory.steps[0].task
                input_for_summarizer = transform_memory(evaluator_agent.memory.steps[1:], i - 3, previous_summary)

                time.sleep(5)
                generated_summary = memory_summarizer_agent.run(input_for_summarizer)

                evaluator_agent.memory.reset()
                evaluator_agent.memory.steps = [TaskStep(task=generated_summary, task_images=[])]

        # Calculate statistics
        avg_pers = np.mean(pers_scores)
        avg_emot = np.mean(emot_scores)
        avg_share = np.mean(share_scores)
        std_pers = np.std(pers_scores)
        std_emot = np.std(emot_scores)
        std_share = np.std(share_scores)
        
        # Starting the refinement by aggregating feedback through the mediator agent
        input_for_mediator = f"""
1. A dictionary of all good points for each KPI:

{all_good_points}

2. A dictionary of all bad points for each KPI:

{all_bad_points}
"""
        top_5_good_points = {"Persuasiveness": [], "Emotional_Engagement": [], "Shareability": []}
        top_5_bad_points  = {"Persuasiveness": [], "Emotional_Engagement": [], "Shareability": []}

        flag = True
        j = 0
        while flag and (j < 3):
            try:
                time.sleep(2)
                meditaor_response_str = mediator_agent.run(input_for_mediator)
                mediator_response_json = json.loads(meditaor_response_str)
                flag = False
            except:
                print("Response was probably not in JSON format, trying again.")
                j += 1
        

        for kpi in ["Persuasiveness", "Emotional_Engagement", "Shareability"]:
            top_5_good_points[kpi].extend(mediator_response_json[kpi]["GoodPoints"])

        for kpi in ["Persuasiveness", "Emotional_Engagement", "Shareability"]:
            top_5_bad_points[kpi].extend(mediator_response_json[kpi]["BadPoints"])

        text_good_points = ""
        for kpi, points_list in top_5_good_points.items():
            text_good_points += f"{kpi}:\n"
            for j, point in enumerate(points_list, start=1):
                text_good_points += f"{j}. {point}\n"
            text_good_points += "\n"  # blank line after each KPI

        text_bad_points = ""
        for kpi, points_list in top_5_bad_points.items():
            text_bad_points += f"{kpi}:\n"
            for j, point in enumerate(points_list, start=1):
                text_bad_points += f"{j}. {point}\n"
            text_bad_points += "\n"  # blank line after each KPI

        score_statistics = f"""
Persuasiveness:
Average: {avg_pers}
Standard Deviation: {std_pers}

Emotional Engagement:
Average: {avg_emot}
Standard Deviation: {std_emot}

Shareability:
Average: {avg_share}
Standard Deviation: {std_share}
"""
        
        aggregated_feedback = f"""
Aggregated feedback gathered from evaluator agents:
Top 5 good points for each KPI:
{text_good_points}

Top 5 bad points for each KPI:
{text_bad_points}
"""
        # Given aggregated feedback and score statistics, let's now call the manager agent to design a new system prompt
        input_for_manager = input = f"""
1. Agent name: {agent_name}

2. Agent's current system prompt:
{pro_ukrainian_agent.system_prompt}

3. {aggregated_feedback}

4. KPI statistics:
{score_statistics}

5. Target KPI: {dict_target_kpis[agent_name]}
"""
        time.sleep(10)
        new_system_prompt = manager_agent_type3.run(input_for_manager, reset=False)

        # If this was the 6th refinement, remove the first refinement from memory, so that memory will hold last 5 refinements.
        if len(manager_agent_type3.memory.steps) >= 12:
            manager_agent_type3.memory.steps = manager_agent_type3.memory.steps[2:]

        # Add new row to the DataFrame.
        row = {
            "claim": claim, "counter_narrative": counter_narrative, "avg_pers": avg_pers, "avg_emot": avg_emot, "avg_share": avg_share, "std_pers": std_pers, "std_emot": std_emot, "std_share": std_share, "aggregated_feedback": aggregated_feedback, "updated_system_prompt": new_system_prompt
        }
        new_row_df = pd.DataFrame([row])
        dict_of_dfs[agent_name] = pd.concat([dict_of_dfs[agent_name], new_row_df], ignore_index=True)

        # Update system prompt for usage in next iteration. Not neccesarily saving when there's improvement to escape "local minima".
        pro_ukrainian_agents_system_prompts[agent_name] = new_system_prompt

        # Let's check for early stopping
        if agent_name == "CN_Creator_Agent_1":
            # Early stopping based on persuasvieness
            best_pers_score, best_pers_score_iteration, early_stop_flag = early_stopping_check(patience, best_pers_score, best_pers_score_iteration, avg_pers, i)
            dict_of_flags[agent_name] = early_stop_flag

        elif agent_name == "CN_Creator_Agent_2":
            # Early stopping based on Emotional Engagement
            best_emot_score, best_emot_score_iteration, early_stop_flag = early_stopping_check(patience, best_emot_score, best_emot_score_iteration, avg_emot, i)
            dict_of_flags[agent_name] = early_stop_flag

        else:
            # Early stopping based on Shareability
            best_share_score, best_share_score_iteration, early_stop_flag = early_stopping_check(patience, best_share_score, best_share_score_iteration, avg_share, i)
            dict_of_flags[agent_name] = early_stop_flag
        
        # Save DataFrames
        for key, df in dict_of_dfs.items():
            df.to_csv(f"refinement_per_claim_final_results/{claim_str}/{key}.csv", index=False)

        # After modifying system prompts in this iteration, let's create the modified agents for next iterations
        pro_ukrainian_agents_type3 = create_pro_ukrainian_agents_claude(pro_ukrainian_agents_descriptions, pro_ukrainian_agents_system_prompts, temp=1.0)
        
        # Saving system prompts in a JSON file
        with open(f'Refinement_Per_Claim_Final/{claim_str}/system_prompts_old_combs.json', 'w', encoding='utf-8') as f:
            json.dump(pro_ukrainian_agents_system_prompts, f, ensure_ascii=False, indent=2)

    save_pro_russian_agents_memories(pro_russian_agents_type3, evaluator_agents_folder_path)
    save_manager_agent_memory(manager_agent_type3, manager_folder_path, agent_name)
    manager_agent_type3.memory.reset()